                               #######             
                              #                    
        ############ -#####+.##     ######         |	DataDrivenConstruction.io
        ##+########## .-##+ #-   .+########        |	
        ####     ####.     ..    #      ###        |	Dive into the future of flexibility in processing data 
        ####     .#+#.     #    .# .               |	without using BIM tools!
        ####     -#+#.    #.     .#.               |	
        ####      #+#.   ##-                       |	
        ####     ####        #.  #     -####       |	
        ############  .###+ #. ############        |
        #########.   +#### #+.   ########          |	If you have any questions, concerns or special requests,
                           #+                      |	please contact us at info@datadrivenconstruction.io	
                          #                        
                         #  



In [ ]:
#------------------------------------Input data-----------------------------------

import os
import sys
from pathlib import Path

# Folder of this repository (it holds DDC_WINDOWS_Converters and ddc-tools)
REPO = Path(os.environ.get("DDC_REPO", r"C:\DDC\cad2data-Revit-IFC-DWG"))
sys.path.insert(0, str(REPO / "ddc-tools" / "src"))   # or install once:  pip install "<repo>\ddc-tools"
from ddc_tools import convert

# Path address of the folder where Revit|IFC files are located
path = os.environ.get("DDC_BATCH", r'C:\Revit_Projects\test')
# Converter results and the flat tables (old layout, sheet Project_Data) go to <path>\converted
CONV = os.path.join(path, 'converted')

# Categories to analyse: Revit tables hold OST_ codes, IFC tables hold IFC classes
s_cat = ['OST_Walls', 'IfcWall', 'IfcWallStandardCase']

#-----------------------------------------------------------------------------------------

def convert_and_wait(file_path):
    """Runs the DDC converter for one file (RVTconverter.exe --properties --ifc, or the IFC to Excel converter)
    and builds <name>_rvt.xlsx / <name>_ifc.xlsx in CONV, the same as:  ddc convert <file> --xlsx <table>"""
    stem, ext = os.path.splitext(os.path.basename(file_path))
    kind = 'rvt' if ext.lower() in ('.rvt', '.rfa') else 'ifc'
    res = convert.convert(file_path, work=os.path.join(CONV, f'{stem}_{kind}'),
                          xlsx=os.path.join(CONV, f'{stem}_{kind}.xlsx'))
    print(f"Conversion done for project {stem}: {res['elements']} elements -> {res['xlsx']}")

# Ensure output directory exists
os.makedirs(CONV, exist_ok=True)
# Conversion process from RVT and IFC (a finished result folder is reused)
for file in os.listdir(path):
    if file.lower().endswith(('.ifc', '.rvt')):
        convert_and_wait(os.path.join(path, file))

#-----------------------------------------------------------------------------------------

import pandas as pd
from fpdf import FPDF   # once:  pip install fpdf2

import matplotlib.pyplot as plt


def process_xlsx_files(folder_path, s_cat):
    for filename in os.listdir(folder_path):
        if filename.endswith(".xlsx"):
            file_path = os.path.join(folder_path, filename)

            # Load the XLSX file (sheet Project_Data)
            df = pd.read_excel(file_path, sheet_name="Project_Data")

            # Handle NA/NaN values in 'Category' column
            df['Category'] = df['Category'].fillna('')  # Replace NA/NaN with empty string

            # Calculate sum of volumes and widths for the specified category (regular expression)
            dfq = df[df['Category'].str.match(s_cat)]
            dfq = dfq.groupby('Category')[['Volume', 'Width']].sum()

            if not dfq.empty:
                cat_len = str(round(dfq['Width'].sum(), 3))
                cat_vol = str(round(dfq['Volume'].sum(), 3))
            else:
                cat_len = cat_vol = 'N/A'

            # Inside your loop where you process each file
            plt.figure(figsize=(25, 8))  # Set the desired size here
            pie_chart = df.groupby('Category')['Length'].sum().plot.pie(title='Pie Chart: ' + filename)
            plt.savefig(os.path.join(folder_path, filename.replace('.xlsx', '_pie_chart.png')))
            plt.close()  # Close the plot to free up memory

            # Creating a horizontal bar plot
            plt.figure(figsize=(15, 20))
            barh_plot = df.groupby('Category')['Volume'].sum().plot(kind='barh')
            plt.title('Bar Chart: ' + filename[:-5])
            plt.savefig(os.path.join(folder_path, filename.replace('.xlsx', '_barh_plot.jpeg')))
            plt.close()

            # Creating a PDF report
            pdf = FPDF()
            pdf.add_page()
            pdf.set_font('Helvetica', 'B', 16)
            pdf.cell(190, 8, 'Report for ' + filename[:-5], new_x="LMARGIN", new_y="NEXT")
            pdf.set_font('Helvetica', '', 14)
            pdf.cell(190, 8, 'Sum of volumes for ' + s_cat + ': ' + cat_vol, new_x="LMARGIN", new_y="NEXT")
            pdf.cell(190, 8, 'Sum of widths for ' + s_cat + ': ' + cat_len, new_x="LMARGIN", new_y="NEXT")

            # Add additional data to the PDF as needed
            pdf.output(os.path.join(folder_path, filename.replace('.xlsx', '_report.pdf')))

            print(f"Processed {filename}")

# Call the function with the specific category (regular expression: Revit code or IFC class)
process_xlsx_files(CONV, "OST_Walls|IfcWall")